# Unified Academic Paper Classification Pipeline
### This notebook integrates all v6.0 classifiers into a single pipeline

In [16]:
import numpy as np
import pandas as pd
import re
import joblib
from scipy.sparse import hstack, csr_matrix
import xgboost as xgb
from sklearn.feature_extraction.text import TfidfVectorizer
import warnings
warnings.filterwarnings('ignore')

## 1. Text Preprocessing Function

In [17]:
class MethodologyFeatureExtractor:
    """Extract domain-specific features for methodology classification."""
    
    def __init__(self):
        # Expanded methodology-specific keywords
        self.quant_keywords = [
            # Core quantitative
            'hypothesis', 'statistical', 'significance', 'correlation', 'regression',
            'anova', 't-test', 'chi-square', 'experiment', 'survey', 'questionnaire',
            'sample size', 'random', 'control group', 'variable', 'measurement',
            'reliability', 'validity', 'scale', 'factor analysis', 'sem', 'quantitative',
            'empirical', 'numerical', 'data analysis', 'spss', 'stata', 'coefficient',
            'mean', 'median', 'standard deviation', 'variance', 'distribution',
            'parametric', 'non-parametric', 'null hypothesis', 'alternative hypothesis',
            'confidence interval', 'effect size', 'power analysis', 'sampling',

            # Advanced quantitative
            'multivariate', 'cluster analysis', 'discriminant analysis', 'logistic regression',
            'linear regression', 'multiple regression', 'structural equation modeling',
            'confirmatory factor analysis', 'exploratory factor analysis', 'cronbach',
            'likert scale', 'ordinal', 'nominal', 'interval', 'ratio', 'bootstrapping',
            'cross-validation', 'monte carlo', 'bayesian', 'frequentist', 'p-value'
        ]

        self.qual_keywords = [
            # Core qualitative
            'interview', 'observation', 'ethnography', 'phenomenology', 'grounded theory',
            'case study', 'narrative', 'interpretive', 'thematic analysis', 'coding',
            'theme', 'pattern', 'emergent', 'participant', 'informant', 'respondent',
            'qualitative', 'exploratory', 'descriptive', 'rich description', 'context',
            'meaning', 'experience', 'perspective', 'open-ended', 'semi-structured',
            'in-depth', 'focus group', 'field notes', 'reflexivity', 'saturation',
            'purposive sampling', 'snowball sampling', 'theoretical sampling',
            'content analysis', 'discourse analysis', 'narrative analysis',

            # Advanced qualitative
            'hermeneutic', 'interpretative phenomenological analysis', 'ipa',
            'constant comparative', 'axial coding', 'selective coding', 'open coding',
            'member checking', 'triangulation', 'trustworthiness', 'transferability',
            'dependability', 'confirmability', 'credibility', 'thick description'
        ]

        self.mixed_keywords = [
            # Mixed methods core
            'mixed method', 'triangulation', 'convergent', 'sequential', 'concurrent',
            'transformative', 'pragmatic', 'qual-quant', 'quant-qual', 'integration',
            'complementary', 'expansion', 'both qualitative and quantitative',
            'multi-method', 'multiple methods', 'combined approach', 'mixed design',
            'explanatory sequential', 'exploratory sequential', 'convergent parallel',
            'embedded', 'multiphase', 'mixing', 'merging', 'connecting', 'building',

            # Mixed methods advanced
            'joint display', 'meta-inference', 'mixed methods matrix', 'strand',
            'mixing ratio', 'priority', 'weighting', 'dominant', 'supplemental'
        ]

    def extract_features(self, texts: list) -> pd.DataFrame:
        """Extract comprehensive features from text."""
        features = []

        for text in texts:
            text_lower = text.lower()
            words = text_lower.split()
            word_count = max(len(words), 1)

            # Basic statistics
            feat = {
                'word_count': word_count,
                'avg_word_length': np.mean([len(w) for w in words]) if words else 0,
                'sentence_count': text.count('.') + text.count('!') + text.count('?'),
                'has_numbers': bool(re.search(r'\d', text)),
                'number_density': len(re.findall(r'\d+', text)) / word_count,
                'parentheses_count': text.count('(') + text.count('['),
            }

            # Quantitative indicators
            feat['has_sample_size'] = bool(re.search(r'\b[nN]\s*=\s*\d+', text))
            feat['has_p_value'] = bool(re.search(r'[pP]\s*[<>≤≥]\s*0?\.\d+', text))
            feat['has_statistical_software'] = any(sw in text_lower for sw in ['spss', 'stata', 'sas', 'r software'])

            # Qualitative indicators
            feat['has_interview'] = bool(re.search(r'\binterview', text_lower))
            feat['has_coding'] = bool(re.search(r'\bcoding\b|\bcode\b|\bcoded\b', text_lower))
            feat['has_participants'] = bool(re.search(r'\b(participant|informant|respondent)', text_lower))

            # Mixed methods indicators
            feat['has_mixed_explicit'] = bool(re.search(r'\bmixed\s+method', text_lower))
            feat['has_triangulation'] = bool(re.search(r'\btriangulat', text_lower))
            feat['has_sequential'] = bool(re.search(r'\bsequential\b', text_lower))
            feat['has_both_qual_quant'] = bool(re.search(r'\b(qualitative.*quantitative|quantitative.*qualitative)', text_lower))

            # Keyword counts
            feat['quant_keyword_count'] = sum(kw in text_lower for kw in self.quant_keywords)
            feat['qual_keyword_count'] = sum(kw in text_lower for kw in self.qual_keywords)
            feat['mixed_keyword_count'] = sum(kw in text_lower for kw in self.mixed_keywords)

            # Keyword densities
            feat['quant_keyword_density'] = feat['quant_keyword_count'] / word_count
            feat['qual_keyword_density'] = feat['qual_keyword_count'] / word_count
            feat['mixed_keyword_density'] = feat['mixed_keyword_count'] / word_count

            # Method dominance
            total_methods = feat['quant_keyword_count'] + feat['qual_keyword_count'] + feat['mixed_keyword_count'] + 1
            feat['quant_dominance'] = feat['quant_keyword_count'] / total_methods
            feat['qual_dominance'] = feat['qual_keyword_count'] / total_methods
            feat['mixed_dominance'] = feat['mixed_keyword_count'] / total_methods

            # Co-occurrence indicators
            feat['has_both_indicators'] = int((feat['quant_keyword_count'] > 0) and (feat['qual_keyword_count'] > 0))
            feat['method_diversity'] = int(feat['quant_keyword_count'] > 0) + int(feat['qual_keyword_count'] > 0) + int(feat['mixed_keyword_count'] > 0)

            features.append(feat)

        return pd.DataFrame(features)


In [18]:
def preprocess_text(text: str) -> str:
    """Lowercase the text and normalize punctuation/whitespace for consistency."""
    if pd.isna(text):
        return ""
    
    # Lowercase
    text = text.lower()
    
    # Keep alphanumeric and select punctuation (retain . , ; : - () [] + = * /)
    text = re.sub(r"[^a-zA-Z0-9\s\.\,\;\:\-\(\)\[\]\+\=\*/]", " ", text)
    
    # Normalize whitespace
    text = " ".join(text.split())
    
    return text

## 2. Domain Feature Extractor for Discipline/Methodology


In [19]:
class DomainFeatureExtractor:
    """Extract domain-specific features for discipline and methodology classification."""
    
    def __init__(self):
        # Keyword lists for each discipline
        self.cs_keywords = [
            'algorithm', 'complexity', 'computational', 'performance', 'optimization',
            'parallel', 'distributed', 'concurrent', 'runtime', 'efficiency',
            'data structure', 'graph', 'tree', 'sorting', 'searching',
            # ML/AI
            'machine learning', 'neural network', 'deep learning', 'artificial intelligence',
            'classifier', 'prediction', 'training', 'model', 'dataset',
            # Systems
            'compiler', 'operating system', 'architecture', 'processor', 'memory',
            'cache', 'scheduling', 'thread', 'process', 'kernel',
            # Theory
            'np-complete', 'turing', 'automata', 'formal', 'proof',
            'theorem', 'lemma', 'complexity class', 'big-o', 'asymptotic'
        ]
        
        self.is_keywords = [
            # Core IS
            'management', 'business', 'organization', 'system', 'process',
            'enterprise', 'strategic', 'decision', 'information system', 'implementation',
            # Business focus
            'adoption', 'user', 'stakeholder', 'requirement', 'analysis',
            'erp', 'crm', 'supply chain', 'e-commerce', 'digital transformation',
            # Management
            'governance', 'policy', 'framework', 'methodology', 'best practice',
            'change management', 'project management', 'risk', 'compliance', 'audit',
            # IS specific
            'business intelligence', 'data warehouse', 'decision support', 'knowledge management',
            'business process', 'workflow', 'integration', 'alignment', 'value'
        ]
        
        self.it_keywords = [
            # Core IT
            'network', 'security', 'infrastructure', 'administration', 'deployment',
            'server', 'cloud', 'virtualization', 'configuration', 'maintenance',
            # Security
            'firewall', 'encryption', 'authentication', 'protocol', 'tcp/ip',
            'vulnerability', 'patch', 'penetration', 'intrusion', 'malware',
            # Admin/IT Ops
            'database administration', 'backup', 'disaster recovery', 'monitoring',
            'helpdesk', 'support', 'troubleshooting', 'installation', 'upgrade',
            # IT specific
            'vpn', 'dns', 'dhcp', 'active directory', 'ldap',
            'load balancer', 'router', 'switch', 'vlan', 'subnet',
            'windows server', 'linux administration', 'devops', 'container', 'kubernetes'
        ]
        
        # Research method indicators (used in methodology classification)
        self.empirical_keywords = [
            'empirical', 'experiment', 'study', 'survey', 'interview',
            'case study', 'observation', 'data collection', 'analysis', 'evaluation'
        ]
        
        self.theoretical_keywords = [
            'theoretical', 'model', 'framework', 'propose', 'conceptual',
            'formal', 'proof', 'theorem', 'axiom', 'hypothesis'
        ]
        
        # Technical content patterns
        self.technical_patterns = {
            'code_snippet': r'(def |class |function |if |for |while |import |include)',
            'math_notation': r'[∈∀∃∧∨¬→↔⊆⊂∪∩∑∏∫]',
            'equations': r'[=\+\-\*/\^<>≤≥≠]',
            'version_numbers': r'\b\d+\.\d+(\.\d+)?\b',
            'acronyms': r'\b[A-Z]{2,}\b'
        }
    
    def extract_features(self, texts: list) -> pd.DataFrame:
        """Extract features from each text in the list and return a DataFrame."""
        features = []
        
        for text in texts:
            text_lower = text.lower()
            words = text_lower.split()
            word_count = max(len(words), 1)  # avoid division by zero
            
            # Basic text statistics
            feat = {
                'word_count': word_count,
                'avg_word_length': np.mean([len(w) for w in words]) if words else 0,
                'sentence_count': text.count('.') + text.count('!') + text.count('?'),
                'paragraph_count': text.count('\n\n') + 1,
                'has_numbers': bool(re.search(r'\d', text)),
                'number_density': len(re.findall(r'\d+', text)) / word_count,
                'has_urls': bool(re.search(r'http[s]?://', text_lower)),
                'parentheses_count': text.count('(') + text.count('['),
                'quote_count': text.count('"') + text.count("'")
            }
            
            # Technical pattern flags and counts
            for pattern_name, pattern in self.technical_patterns.items():
                feat[f'has_{pattern_name}'] = bool(re.search(pattern, text))
                feat[f'{pattern_name}_count'] = len(re.findall(pattern, text))
            
            # Keyword counts and densities for CS, IS, IT and research methods
            for keyword_type, keywords in [
                ('cs', self.cs_keywords),
                ('is', self.is_keywords),
                ('it', self.it_keywords),
                ('empirical', self.empirical_keywords),
                ('theoretical', self.theoretical_keywords)
            ]:
                matches = sum(kw in text_lower for kw in keywords)
                feat[f'{keyword_type}_keyword_count'] = matches
                feat[f'{keyword_type}_keyword_density'] = matches / word_count
                
                # Partial (substring) matches for keywords
                partial_matches = sum(any(kw in word for word in words) for kw in keywords)
                feat[f'{keyword_type}_partial_matches'] = partial_matches
            
            # Keyword ratios across CS/IS/IT (if any technical keywords found)
            total_tech_kw = feat['cs_keyword_count'] + feat['is_keyword_count'] + feat['it_keyword_count']
            if total_tech_kw > 0:
                feat['cs_ratio'] = feat['cs_keyword_count'] / total_tech_kw
                feat['is_ratio'] = feat['is_keyword_count'] / total_tech_kw
                feat['it_ratio'] = feat['it_keyword_count'] / total_tech_kw
            else:
                feat['cs_ratio'] = feat['is_ratio'] = feat['it_ratio'] = 0
            
            # Special compound indicators
            feat['has_case_study'] = int('case study' in text_lower or 'case-study' in text_lower)
            feat['has_algorithm'] = int('algorithm' in text_lower or 'algo' in text_lower)
            feat['has_framework'] = int('framework' in text_lower or 'model' in text_lower)
            feat['has_implementation'] = int('implement' in text_lower or 'develop' in text_lower)
            feat['has_evaluation'] = int('evaluat' in text_lower or 'assess' in text_lower)
            feat['has_comparison'] = int('compar' in text_lower or ' versus ' in text_lower or ' vs ' in text_lower)
            
            # First sentence (often contains key indicators, akin to a title or leading sentence)
            first_sentence = text.split('.')[0] if '.' in text else text[:100]
            feat['first_sentence_cs_keywords'] = sum(kw in first_sentence.lower() for kw in self.cs_keywords[:10])
            feat['first_sentence_is_keywords'] = sum(kw in first_sentence.lower() for kw in self.is_keywords[:10])
            feat['first_sentence_it_keywords'] = sum(kw in first_sentence.lower() for kw in self.it_keywords[:10])
            
            features.append(feat)
        
        return pd.DataFrame(features)


## 3. Subfield Feature Extractor for each Discipline

In [20]:
class SubfieldFeatureExtractor:
    """Extract domain-specific features for subfield classification (discipline-specific)."""
    
    def __init__(self, discipline: str):
        self.discipline = discipline
        
        # Define subfield-specific keyword lists based on discipline
        if discipline == 'CS':
            self.subfield_keywords = {
                'AI/ML': [
                    'machine learning', 'artificial intelligence', 'neural network',
                    'deep learning', 'reinforcement learning', 'supervised learning',
                    'unsupervised learning', 'classification', 'regression', 'clustering',
                    'training data', 'model performance', 'accuracy', 'precision', 'recall',
                    'tensorflow', 'pytorch', 'scikit-learn', 'gradient descent', 'backpropagation'
                ],
                'SE': [
                    'software engineering', 'software development', 'agile', 'scrum',
                    'requirements engineering', 'software architecture', 'design patterns',
                    'testing', 'debugging', 'version control', 'git', 'continuous integration',
                    'software quality', 'refactoring', 'code review', 'documentation',
                    'software maintenance', 'technical debt', 'software metrics'
                ],
                'SEC': [
                    'security', 'cybersecurity', 'encryption', 'cryptography', 'vulnerability',
                    'threat', 'attack', 'malware', 'intrusion detection', 'firewall',
                    'authentication', 'authorization', 'privacy', 'data protection',
                    'security protocol', 'penetration testing', 'security audit', 'risk assessment'
                ],
                'CV': [
                    'computer vision', 'image processing', 'object detection', 'image recognition',
                    'image segmentation', 'feature extraction', 'convolutional neural network',
                    'cnn', 'image classification', 'face recognition', 'optical character recognition',
                    'ocr', 'video analysis', 'visual tracking', 'image enhancement'
                ],
                'NLP': [
                    'natural language processing', 'text mining', 'sentiment analysis',
                    'named entity recognition', 'ner', 'part of speech', 'pos tagging',
                    'machine translation', 'language model', 'bert', 'transformer',
                    'text classification', 'information extraction', 'word embedding',
                    'tokenization', 'lemmatization', 'text generation'
                ],
                'CLOUDCS': [
                    'cloud computing', 'distributed systems', 'scalability', 'cloud architecture',
                    'microservices', 'serverless', 'cloud native', 'containerization',
                    'orchestration', 'service mesh', 'cloud patterns', 'multi-tenancy',
                    'distributed computing', 'cloud performance', 'cloud optimization',
                    'fog computing', 'edge cloud'
                ]
            }
        elif discipline == 'IS':
            self.subfield_keywords = {
                'DT': [
                    'digital transformation', 'digitalization', 'digital strategy',
                    'digital innovation', 'digital business', 'digital economy',
                    'digital disruption', 'digital maturity', 'digital capability',
                    'digital culture', 'digital leadership', 'industry 4.0',
                    'digital platform', 'digital ecosystem', 'digital value'
                ],
                'KM': [
                    'knowledge management', 'knowledge sharing', 'knowledge transfer',
                    'tacit knowledge', 'explicit knowledge', 'knowledge creation',
                    'organizational learning', 'intellectual capital', 'knowledge repository',
                    'best practices', 'lessons learned', 'expertise', 'knowledge base',
                    'communities of practice', 'knowledge capture'
                ],
                'BPM': [
                    'business process', 'process management', 'workflow', 'process optimization',
                    'process improvement', 'process modeling', 'bpmn', 'process automation',
                    'process mining', 'process analysis', 'process redesign', 'process performance',
                    'business process reengineering', 'bpr', 'process efficiency'
                ],
                'HIS': [
                    'health information', 'healthcare system', 'electronic health record',
                    'ehr', 'emr', 'clinical decision support', 'telemedicine', 'e-health',
                    'health it', 'patient data', 'medical informatics', 'healthcare analytics',
                    'clinical information', 'health data', 'hospital information system'
                ],
                'GOV': [
                    'e-government', 'digital government', 'government service', 'public administration',
                    'citizen engagement', 'public policy', 'government transformation',
                    'smart city', 'open government', 'government portal', 'public sector',
                    'civic technology', 'government efficiency', 'digital democracy'
                ]
            }
        elif discipline == 'IT':
            self.subfield_keywords = {
                'CLOUDIT': [
                    'cloud infrastructure', 'cloud service', 'iaas', 'paas', 'saas',
                    'cloud migration', 'cloud security', 'cloud storage', 'cloud deployment',
                    'aws', 'azure', 'google cloud', 'cloud management', 'cloud monitoring',
                    'virtualization', 'cloud administration', 'hybrid cloud', 'multi-cloud',
                    'cloud backup', 'cloud disaster recovery', 'cloud cost optimization'
                ],
                'DEVOPS': [
                    'devops', 'continuous integration', 'continuous deployment', 'ci/cd',
                    'infrastructure as code', 'automation', 'docker', 'kubernetes', 'container',
                    'microservices', 'monitoring', 'logging', 'configuration management',
                    'ansible', 'terraform', 'jenkins', 'gitlab', 'deployment pipeline'
                ],
                'EMERGING': [
                    'internet of things', 'iot', 'edge computing', 'fog computing',
                    'blockchain', 'distributed ledger', '5g', 'quantum computing',
                    'augmented reality', 'virtual reality', 'ar/vr', 'wearable',
                    'smart device', 'sensor network', 'emerging technology'
                ],
                'RISK': [
                    'risk management', 'it risk', 'risk assessment', 'risk mitigation',
                    'business continuity', 'disaster recovery', 'incident response',
                    'compliance', 'governance', 'risk framework', 'vulnerability management',
                    'threat intelligence', 'risk analysis', 'security risk', 'operational risk'
                ]
            }
        
        # Common technical patterns (similar to DomainFeatureExtractor)
        self.technical_patterns = {
            'code_snippet': r'(def |class |function |if |for |while |import |include)',
            'math_notation': r'[∈∀∃∧∨¬→↔⊆⊂∪∩∑∏∫]',
            'equations': r'[=\+\-\*/\^<>≤≥≠]',
            'version_numbers': r'\b\d+\.\d+(\.\d+)?\b',
            'acronyms': r'\b[A-Z]{2,}\b',
            'urls': r'http[s]?://',
            'citations': r'\[\d+\]|\(\d{4}\)'
        }
    
    def extract_features(self, texts: list) -> pd.DataFrame:
        """Extract features for subfield classification."""
        features = []
        
        for text in texts:
            text_lower = text.lower()
            words = text_lower.split()
            word_count = max(len(words), 1)
            
            feat = {
                'word_count': word_count,
                'avg_word_length': np.mean([len(w) for w in words]) if words else 0,
                'sentence_count': text.count('.') + text.count('!') + text.count('?'),
                'paragraph_count': text.count('\n\n') + 1,
                'unique_word_ratio': len(set(words)) / word_count if words else 0,
                'long_word_count': sum(1 for w in words if len(w) > 10),
                'technical_term_density': sum(1 for w in words if len(w) > 6) / word_count
            }
            
            # Technical pattern features
            for pattern_name, pattern in self.technical_patterns.items():
                feat[f'has_{pattern_name}'] = bool(re.search(pattern, text))
                feat[f'{pattern_name}_count'] = len(re.findall(pattern, text))
            
            # Subfield keyword features (counts, densities, presence in first sentence)
            for subfield, keywords in self.subfield_keywords.items():
                matches = sum(kw in text_lower for kw in keywords)
                feat[f'{subfield}_keyword_count'] = matches
                feat[f'{subfield}_keyword_density'] = matches / word_count
                
                exact_matches = sum(f" {kw} " in f" {text_lower} " for kw in keywords)
                feat[f'{subfield}_exact_matches'] = exact_matches
                
                # First sentence/title indicators (first sentence of text)
                first_sentence = text.split('.')[0].lower() if '.' in text else text[:200].lower()
                feat[f'{subfield}_in_title'] = sum(kw in first_sentence for kw in keywords[:5])
            
            # Keyword dominance ratios (relative frequency among subfield keywords)
            total_keywords = sum(feat[f'{sf}_keyword_count'] for sf in self.subfield_keywords.keys())
            if total_keywords > 0:
                for subfield in self.subfield_keywords.keys():
                    feat[f'{subfield}_dominance'] = feat[f'{subfield}_keyword_count'] / total_keywords
            else:
                for subfield in self.subfield_keywords.keys():
                    feat[f'{subfield}_dominance'] = 0
            
            # Special indicators by discipline context
            if self.discipline == 'CS':
                feat['has_algorithm'] = int('algorithm' in text_lower or 'algorithmic' in text_lower)
                feat['has_implementation'] = int('implement' in text_lower or 'develop' in text_lower)
                feat['has_evaluation'] = int('evaluat' in text_lower or 'experiment' in text_lower)
                feat['has_performance'] = int('performance' in text_lower or 'efficiency' in text_lower)
            elif self.discipline == 'IS':
                feat['has_case_study'] = int('case study' in text_lower or 'case-study' in text_lower)
                feat['has_framework'] = int('framework' in text_lower or 'model' in text_lower)
                feat['has_organization'] = int('organization' in text_lower or 'enterprise' in text_lower)
                feat['has_management'] = int('management' in text_lower or 'manage' in text_lower)
            elif self.discipline == 'IT':
                feat['has_deployment'] = int('deploy' in text_lower or 'implement' in text_lower)
                feat['has_infrastructure'] = int('infrastructure' in text_lower or 'system' in text_lower)
                feat['has_configuration'] = int('configur' in text_lower or 'setup' in text_lower)
                feat['has_administration'] = int('admin' in text_lower or 'manage' in text_lower)
            
            features.append(feat)
        
        return pd.DataFrame(features)


## 4. Load Pre-trained Classifier Models


In [21]:
import os

# Check current directory and find model files
print("Current directory:", os.getcwd())
print("\nChecking for model files...")

# Try different possible paths
possible_paths = ['Artefacts/', '../Artefacts/', './', '../../Artefacts/']
found_path = None

for path in possible_paths:
    if os.path.exists(path) and os.path.exists(f'{path}discipline_classifier_v6.0/'):
        found_path = path
        print(f"✅ Found models in: {path}")
        break

if not found_path:
    print("❌ Could not find models directory.")
    print("\nExpected structure:")
    print("  Artefacts/")
    print("    ├── discipline_classifier_v6.0/")
    print("    ├── methodology_classifier_v6.0/")
    print("    └── subfield_classifier_v6.0/")
else:
    # Load the pre-trained pipeline artifacts
    print("\nLoading pre-trained models...")
    
    discipline_pipeline = joblib.load(f"{found_path}discipline_classifier_v6.0/discipline_classifier_v6.0_pipeline.pkl")
    cs_subfield_pipeline = joblib.load(f"{found_path}subfield_classifier_v6.0/cs_subfield_classifier_v6.0/cs_subfield_classifier_v6.0_pipeline.pkl")
    is_subfield_pipeline = joblib.load(f"{found_path}subfield_classifier_v6.0/is_subfield_classifier_v6.0/is_subfield_classifier_v6.0_pipeline.pkl")
    it_subfield_pipeline = joblib.load(f"{found_path}subfield_classifier_v6.0/it_subfield_classifier_v6.0/it_subfield_classifier_v6.0_pipeline.pkl")
    methodology_pipeline = joblib.load(f"{found_path}methodology_classifier_v6.0/methodology_classifier_v6.0_pipeline.pkl")
    
    # Extract class label mappings
    discipline_labels = discipline_pipeline['label_encoder'].classes_
    methodology_labels = methodology_pipeline['label_encoder'].classes_
    
    print("\n✅ Models loaded successfully!")
    print(f"Available discipline classes: {discipline_labels}")
    print(f"Available methodology classes: {methodology_labels}")


Current directory: /Users/aanandprabhu/Desktop/development/NLP-Project/Notebooks

Checking for model files...
✅ Found models in: ../Artefacts/

Loading pre-trained models...

✅ Models loaded successfully!
Available discipline classes: ['CS' 'IS' 'IT']
Available methodology classes: ['Mixed' 'Qualitative' 'Quantitative']


## 5. Hierarchical Classification Function

In [22]:
def classify_paper(title: str, abstract: str, confidence_threshold: float = 0.6):
    """
    Classify a research paper into Discipline, Subfield, and Methodology.
    Includes smart post-processing for low-confidence predictions.
    
    Args:
        title: Paper title
        abstract: Paper abstract
        confidence_threshold: Minimum confidence for accepting model prediction (default: 0.6)
    
    Returns:
        dict: Predictions and confidences with correction flags
    """
    # Combine title and abstract for analysis
    combined_text = title.strip()
    if combined_text and combined_text[-1] not in ".!?":
        combined_text += "."
    combined_text += " " + abstract.strip()
    
    # Preprocess text
    processed_text = preprocess_text(combined_text)
    
    # 1. Discipline Prediction
    # Extract features using the discipline pipeline's feature extractor and vectorizers
    disc_feat_extractor = discipline_pipeline['feature_extractor']
    disc_domain_feats = disc_feat_extractor.extract_features([processed_text])
    
    # TF-IDF features for discipline (multiple vectorizers combined)
    disc_tfidf_feats = []
    for name, vec in discipline_pipeline['tfidf_vectorizers'].items():
        disc_tfidf_feats.append(vec.transform([processed_text]))
    
    X_disc = hstack(disc_tfidf_feats + [csr_matrix(disc_domain_feats.values, dtype=float)])
    
    # Predict discipline label and confidence
    disc_model = discipline_pipeline['ensemble_models'][0]
    disc_pred_encoded = disc_model.predict(X_disc)[0]
    disc_pred_label = discipline_pipeline['label_encoder'].inverse_transform([disc_pred_encoded])[0]
    disc_proba = disc_model.predict_proba(X_disc)[0]
    disc_conf = float(np.max(disc_proba))
    
    # Smart Post-Processing for Low-Confidence Discipline Predictions
    original_disc_label = disc_pred_label
    correction_applied = False
    
    if disc_conf < confidence_threshold:
        # Check domain features for strong indicators
        cs_count = disc_domain_feats.iloc[0]['cs_keyword_count']
        is_count = disc_domain_feats.iloc[0]['is_keyword_count'] 
        it_count = disc_domain_feats.iloc[0]['it_keyword_count']
        cs_ratio = disc_domain_feats.iloc[0]['cs_ratio']
        
        # Rule-based correction for obvious CS papers
        strong_cs_indicators = [
            'machine learning', 'deep learning', 'neural network', 'artificial intelligence',
            'computer vision', 'natural language processing', 'algorithm', 'computational'
        ]
        
        strong_ai_ml_terms = [
            'convolutional neural network', 'u-net', 'transformer', 'bert', 'cnn', 'rnn',
            'tensorflow', 'pytorch', 'gradient descent', 'backpropagation'
        ]
        
        text_lower = processed_text.lower()
        
        # Check for strong CS/AI indicators
        cs_strong_matches = sum(1 for term in strong_cs_indicators if term in text_lower)
        ai_ml_matches = sum(1 for term in strong_ai_ml_terms if term in text_lower)
        
        # Correction logic
        if (cs_count >= 3 and cs_ratio > 0.6) or cs_strong_matches >= 2 or ai_ml_matches >= 1:
            if disc_pred_label != 'CS':
                disc_pred_label = 'CS'
                disc_conf = 0.85  # Assign high confidence for rule-based correction
                correction_applied = True
        
        # Similar logic for IT papers with strong infrastructure indicators
        elif 'cloud' in text_lower and ('infrastructure' in text_lower or 'security' in text_lower):
            if disc_pred_label != 'IT':
                disc_pred_label = 'IT'
                disc_conf = 0.80
                correction_applied = True
    
    # 2. Subfield Prediction (choose the correct subfield model based on discipline)
    if disc_pred_label == 'CS':
        sub_pipeline = cs_subfield_pipeline
    elif disc_pred_label == 'IS':
        sub_pipeline = is_subfield_pipeline
    else:  # 'IT'
        sub_pipeline = it_subfield_pipeline
    
    sub_feat_extractor = sub_pipeline['feature_extractor']
    sub_domain_feats = sub_feat_extractor.extract_features([processed_text])
    
    sub_tfidf_feats = []
    for name, vec in sub_pipeline['tfidf_vectorizers'].items():
        sub_tfidf_feats.append(vec.transform([processed_text]))
    
    X_sub = hstack(sub_tfidf_feats + [csr_matrix(sub_domain_feats.values, dtype=float)])
    
    sub_model = sub_pipeline['model']
    sub_pred_encoded = sub_model.predict(X_sub)[0]
    sub_pred_label = sub_pipeline['label_encoder'].inverse_transform([sub_pred_encoded])[0]
    sub_proba = sub_model.predict_proba(X_sub)[0]
    sub_conf = float(np.max(sub_proba))
    
    # 3. Methodology Prediction
    # Create a methodology-specific feature extractor that matches the training data
    meth_feat_extractor = MethodologyFeatureExtractor()
    meth_domain_feats = meth_feat_extractor.extract_features([processed_text])
    
    meth_tfidf_feats = []
    for name, vec in methodology_pipeline['tfidf_vectorizers'].items():
        meth_tfidf_feats.append(vec.transform([processed_text]))
    
    X_meth = hstack(meth_tfidf_feats + [csr_matrix(meth_domain_feats.values, dtype=float)])
    
    # The methodology pipeline may contain an ensemble of models; we use the first (or only) model
    meth_model = methodology_pipeline['ensemble_models'][0]
    meth_pred_encoded = meth_model.predict(X_meth)[0]
    meth_pred_label = methodology_pipeline['label_encoder'].inverse_transform([meth_pred_encoded])[0]
    meth_proba = meth_model.predict_proba(X_meth)[0]
    meth_conf = float(np.max(meth_proba))
    
    # Print results in a readable format
    print(f"\n{'='*60}")
    print(f"Classification Results")
    print(f"{'='*60}")
    
    if correction_applied:
        print(f"Discipline: **{disc_pred_label}** (confidence {disc_conf*100:.1f}%) 🔧")
        print(f"  ↳ Corrected from '{original_disc_label}' due to low confidence + strong indicators")
    else:
        print(f"Discipline: **{disc_pred_label}** (confidence {disc_conf*100:.1f}%)")
    
    print(f"Subfield ({disc_pred_label}): **{sub_pred_label}** (confidence {sub_conf*100:.1f}%)")
    print(f"Methodology: **{meth_pred_label}** (confidence {meth_conf*100:.1f}%)")
    print(f"{'='*60}")
    
    # Also return a dictionary of results
    return {
        "discipline": disc_pred_label,
        "discipline_confidence": disc_conf,
        "discipline_corrected": correction_applied,
        "original_discipline": original_disc_label if correction_applied else disc_pred_label,
        "subfield": sub_pred_label,
        "subfield_confidence": sub_conf,
        "methodology": meth_pred_label,
        "methodology_confidence": meth_conf
    }


In [23]:
def analyze_classification_confidence(results: dict, title: str, abstract: str):
    """
    Provide detailed analysis and recommendations for classification results.
    
    Args:
        results: Results from classify_paper()
        title: Original paper title
        abstract: Original paper abstract
    """
    print(f"\n📊 Classification Analysis & Recommendations")
    print(f"{'='*50}")
    
    # Overall confidence assessment
    disc_conf = results['discipline_confidence']
    sub_conf = results['subfield_confidence'] 
    meth_conf = results['methodology_confidence']
    
    avg_confidence = (disc_conf + sub_conf + meth_conf) / 3
    
    print(f"Overall Confidence: {avg_confidence*100:.1f}%")
    
    if avg_confidence >= 0.8:
        print("✅ HIGH CONFIDENCE - Results are highly reliable")
    elif avg_confidence >= 0.6:
        print("⚠️  MEDIUM CONFIDENCE - Results are generally reliable")
    else:
        print("❌ LOW CONFIDENCE - Manual review recommended")
    
    print()
    
    # Individual component analysis
    confidence_levels = [
        ("Discipline", disc_conf, results['discipline']),
        ("Subfield", sub_conf, results['subfield']),
        ("Methodology", meth_conf, results['methodology'])
    ]
    
    for component, conf, prediction in confidence_levels:
        status = "🟢" if conf >= 0.7 else "🟡" if conf >= 0.5 else "🔴"
        print(f"{status} {component}: {prediction} ({conf*100:.1f}%)")
        
        if conf < 0.5:
            print(f"   ⚠️  Very low confidence - consider manual verification")
        elif conf < 0.7:
            print(f"   ℹ️  Moderate confidence - review recommended")
    
    print()
    
    # Special notes
    if results.get('discipline_corrected', False):
        print("🔧 CORRECTION APPLIED:")
        print(f"   Original model predicted: {results['original_discipline']}")
        print(f"   Corrected to: {results['discipline']}")
        print("   Reason: Low model confidence + strong domain indicators")
        print()
    
    # Recommendations
    print("💡 Recommendations:")
    
    if avg_confidence < 0.6:
        print("   • Manual review strongly recommended")
        print("   • Consider consulting domain experts")
    
    if disc_conf < 0.6:
        print("   • Verify discipline classification with paper content")
        
    if sub_conf < 0.6:
        print("   • Subfield may be emerging/interdisciplinary")
        
    if meth_conf < 0.6:
        print("   • Check paper for mixed methodologies")
    
    # Paper characteristics that might affect classification
    text_combined = f"{title} {abstract}".lower()
    
    if 'medical' in text_combined or 'healthcare' in text_combined:
        print("   • Medical/healthcare context detected - domain boundaries may be blurred")
    
    if len(text_combined.split()) < 50:
        print("   • Short abstract - limited context for classification")
    
    if 'novel' in text_combined or 'new' in text_combined:
        print("   • Novel approach detected - may represent emerging research areas")


## 6. Example Usage

In [24]:
if 'discipline_pipeline' in locals():
    # Example paper (from the PDF)
    example_title = "A Novel Approach to Cloud Infrastructure Security"
    example_abstract = (
        "Cloud computing has become integral to IT services. This paper presents "
        "an improved method for detecting configuration vulnerabilities in cloud infrastructure. "
        "We conduct an empirical evaluation across different platforms, comparing our approach "
        "with existing security frameworks. The results demonstrate significant "
        "improvements in threat detection rates under various conditions."
    )
    
    # Run the classification
    results = classify_paper(example_title, example_abstract)
    analyze_classification_confidence(results, example_title, example_abstract)
    
    # Additional test cases
    print("\n\n" + "="*80)
    print("TESTING WITH MORE EXAMPLES")
    print("="*80)
    
    # CS Example (previously misclassified)
    print("\n🧪 Test Case 1: Computer Science Paper")
    cs_title = "Deep Learning for Medical Image Segmentation"
    cs_abstract = """This paper presents a novel deep learning architecture for segmenting 
    medical images using convolutional neural networks. We employ a U-Net based approach 
    with attention mechanisms to improve segmentation accuracy on CT scans. Our quantitative 
    experiments on a dataset of 1000 CT images show significant improvements over baseline 
    methods, achieving a Dice coefficient of 0.92."""
    
    results_cs = classify_paper(cs_title, cs_abstract)
    analyze_classification_confidence(results_cs, cs_title, cs_abstract)
    
    # IS Example
    print("\n\n🧪 Test Case 2: Information Systems Paper")
    is_title = "Digital Transformation in Healthcare Organizations"
    is_abstract = """This qualitative study examines the digital transformation journey of 
    three major healthcare organizations. Through semi-structured interviews with 45 
    executives and IT managers, we identify key success factors and barriers to digital 
    adoption. Our thematic analysis reveals that organizational culture and leadership 
    commitment are critical factors in successful digital transformation initiatives."""
    
    results_is = classify_paper(is_title, is_abstract)
    analyze_classification_confidence(results_is, is_title, is_abstract)
    
    print("\n" + "="*80)
    print("SUMMARY OF IMPROVEMENTS")
    print("="*80)
    print("✅ Added confidence-based post-processing")
    print("✅ Smart correction for obvious misclassifications") 
    print("✅ Detailed analysis and recommendations")
    print("✅ Transparency in correction decisions")
    print("\n💡 The system now handles edge cases and provides actionable insights!")
else:
    print("⚠️  Models not loaded. Please check the model files path.")



Classification Results
Discipline: **IT** (confidence 70.8%)
Subfield (IT): **CLOUDIT** (confidence 73.1%)
Methodology: **Quantitative** (confidence 99.3%)

📊 Classification Analysis & Recommendations
Overall Confidence: 81.0%
✅ HIGH CONFIDENCE - Results are highly reliable

🟢 Discipline: IT (70.8%)
🟢 Subfield: CLOUDIT (73.1%)
🟢 Methodology: Quantitative (99.3%)

💡 Recommendations:
   • Novel approach detected - may represent emerging research areas


TESTING WITH MORE EXAMPLES

🧪 Test Case 1: Computer Science Paper

Classification Results
Discipline: **CS** (confidence 85.0%) 🔧
  ↳ Corrected from 'IS' due to low confidence + strong indicators
Subfield (CS): **CV** (confidence 95.6%)
Methodology: **Quantitative** (confidence 98.3%)

📊 Classification Analysis & Recommendations
Overall Confidence: 93.0%
✅ HIGH CONFIDENCE - Results are highly reliable

🟢 Discipline: CS (85.0%)
🟢 Subfield: CV (95.6%)
🟢 Methodology: Quantitative (98.3%)

🔧 CORRECTION APPLIED:
   Original model predicted: I

## 7. Batch Classification Function

In [25]:
def classify_papers_batch(papers_list):
    """
    Classify multiple papers at once.
    
    Args:
        papers_list: List of dictionaries with 'title' and 'abstract' keys
    
    Returns:
        List of classification results
    """
    results = []
    
    for i, paper in enumerate(papers_list, 1):
        print(f"\nProcessing paper {i}/{len(papers_list)}...")
        result = classify_paper(paper['title'], paper['abstract'])
        results.append(result)
    
    return results


In [26]:
# Example batch processing
if 'discipline_pipeline' in locals():
    batch_papers = [
        {
            'title': 'Implementing DevOps Practices in Cloud Infrastructure',
            'abstract': '''This paper describes the implementation of DevOps practices in a 
            large-scale cloud infrastructure environment. We present a mixed-methods approach 
            combining quantitative metrics from CI/CD pipelines with qualitative insights from 
            developer surveys. Our findings show a 40% reduction in deployment time and improved 
            team collaboration after adopting Infrastructure as Code and automated testing.'''
        },
        {
            'title': 'Knowledge Management Systems in Digital Organizations',
            'abstract': '''This research investigates how organizations leverage knowledge management 
            systems to enhance digital capabilities. Through case studies of five Fortune 500 companies, 
            we analyze the implementation of knowledge repositories, best practice databases, and 
            collaborative platforms. Our findings suggest that successful knowledge management requires 
            both technological infrastructure and cultural transformation.'''
        }
    ]
    
    batch_results = classify_papers_batch(batch_papers)


Processing paper 1/2...

Classification Results
Discipline: **IT** (confidence 54.2%)
Subfield (IT): **DEVOPS** (confidence 66.2%)
Methodology: **Quantitative** (confidence 59.3%)

Processing paper 2/2...

Classification Results
Discipline: **IS** (confidence 92.7%)
Subfield (IS): **DT** (confidence 92.2%)
Methodology: **Quantitative** (confidence 65.3%)


## 8. Model Performance Summary

In [27]:
print("\n" + "="*60)
print("Model Performance Summary (v6.0)")
print("="*60)
print("\nDiscipline Classifier:")
print("  - Accuracy: 94.77%")
print("  - Macro F1: 0.9483")
print("  - Features: 11,546 (11,500 TF-IDF + 46 domain-specific)")
print("  - Per-class: CS=92.60%, IS=94.86%, IT=97.27%")

print("\nMethodology Classifier:")
print("  - Accuracy: 91.87%")
print("  - Macro F1: 0.9180")
print("  - Features: 11,527 (11,500 TF-IDF + 27 domain-specific)")
print("  - Per-class: Mixed=88.78%, Qual=91.16%, Quant=94.81%")

print("\nSubfield Classifiers:")
print("  - CS Subfield: 89.61% accuracy")
print("    Features: 11,555 (11,500 TF-IDF + 55 domain-specific)")
print("    Classes: AI/ML, CLOUDCS, CV, NLP, SE, SEC")
print("  - IS Subfield: 83.39% accuracy")
print("    Features: 11,550 (11,500 TF-IDF + 50 domain-specific)")
print("    Classes: BPM, DT, GOV, HIS, KM")
print("  - IT Subfield: 88.48% accuracy")
print("    Features: 11,545 (11,500 TF-IDF + 45 domain-specific)")
print("    Classes: CLOUDIT, DEVOPS, EMERGING, RISK")
print("="*60)


Model Performance Summary (v6.0)

Discipline Classifier:
  - Accuracy: 94.77%
  - Macro F1: 0.9483
  - Features: 11,546 (11,500 TF-IDF + 46 domain-specific)
  - Per-class: CS=92.60%, IS=94.86%, IT=97.27%

Methodology Classifier:
  - Accuracy: 91.87%
  - Macro F1: 0.9180
  - Features: 11,527 (11,500 TF-IDF + 27 domain-specific)
  - Per-class: Mixed=88.78%, Qual=91.16%, Quant=94.81%

Subfield Classifiers:
  - CS Subfield: 89.61% accuracy
    Features: 11,555 (11,500 TF-IDF + 55 domain-specific)
    Classes: AI/ML, CLOUDCS, CV, NLP, SE, SEC
  - IS Subfield: 83.39% accuracy
    Features: 11,550 (11,500 TF-IDF + 50 domain-specific)
    Classes: BPM, DT, GOV, HIS, KM
  - IT Subfield: 88.48% accuracy
    Features: 11,545 (11,500 TF-IDF + 45 domain-specific)
    Classes: CLOUDIT, DEVOPS, EMERGING, RISK


## 9. Export Functions for Integration


In [28]:
def save_classification_results(results, filename='classification_results.json'):
    """Save classification results to JSON file"""
    import json
    with open(filename, 'w') as f:
        json.dump(results, f, indent=2)
    print(f"Results saved to {filename}")

def create_classification_report(papers_with_results):
    """Create a summary report of classification results"""
    df_results = pd.DataFrame(papers_with_results)
    
    # Summary statistics
    print("\nClassification Summary Report")
    print("="*60)
    print(f"Total papers classified: {len(df_results)}")
    print("\nDiscipline Distribution:")
    print(df_results['discipline'].value_counts())
    print("\nMethodology Distribution:")
    print(df_results['methodology'].value_counts())
    
    return df_results

## 10. Production-Ready API Wrapper

In [29]:
class AcademicPaperClassifier:
    """Production-ready wrapper for the academic paper classification pipeline"""
    
    def __init__(self, models_path='../Artefacts/'):
        self.models_loaded = False
        try:
            # Load all pipelines
            self.discipline_pipeline = joblib.load(f"{models_path}discipline_classifier_v6.0/discipline_classifier_v6.0_pipeline.pkl")
            self.cs_subfield_pipeline = joblib.load(f"{models_path}subfield_classifier_v6.0/cs_subfield_classifier_v6.0/cs_subfield_classifier_v6.0_pipeline.pkl")
            self.is_subfield_pipeline = joblib.load(f"{models_path}subfield_classifier_v6.0/is_subfield_classifier_v6.0/is_subfield_classifier_v6.0_pipeline.pkl")
            self.it_subfield_pipeline = joblib.load(f"{models_path}subfield_classifier_v6.0/it_subfield_classifier_v6.0/it_subfield_classifier_v6.0_pipeline.pkl")
            self.methodology_pipeline = joblib.load(f"{models_path}methodology_classifier_v6.0/methodology_classifier_v6.0_pipeline.pkl")
            self.models_loaded = True
            print("✅ All models loaded successfully!")
        except Exception as e:
            print(f"❌ Error loading models: {e}")
    
    def classify(self, title, abstract):
        """Classify a single paper"""
        if not self.models_loaded:
            return {"error": "Models not loaded"}
        
        return classify_paper(title, abstract)
    
    def classify_batch(self, papers_list):
        """Classify multiple papers"""
        if not self.models_loaded:
            return [{"error": "Models not loaded"}] * len(papers_list)
        
        return classify_papers_batch(papers_list)
    
    def get_model_info(self):
        """Get information about loaded models"""
        return {
            "version": "6.0",
            "models_loaded": self.models_loaded,
            "discipline_accuracy": 0.9477,
            "methodology_accuracy": 0.9187,
            "subfield_accuracies": {
                "CS": 0.8961,
                "IS": 0.8339,
                "IT": 0.8848
            }
        }

# Initialize the classifier for API use
if 'discipline_pipeline' in locals():
    api_classifier = AcademicPaperClassifier(models_path=found_path)
    print("\n✅ API Classifier ready for use!")
    print("Usage: api_classifier.classify(title, abstract)")

✅ All models loaded successfully!

✅ API Classifier ready for use!
Usage: api_classifier.classify(title, abstract)
